# Defining a Custom Climate Indicator from Scratch

While **earthkit-climate** provides an extensive catalogue of standard climate indicators (and allows adapting existing indicators via registries or YAML modules), domain-specific applications often require compound or multi-variable indices that are not present in standard catalogues.

This tutorial walks step-by-step through designing, implementing, and validating an operational agrometeorological disease index: the **Fusarium head blight (FHB) conducive-days indicator**.

Fusarium head blight is a destructive fungal disease affecting cereal crops such as wheat, barley, and oats. An infection event requires warm, wet, and highly humid weather during flowering. Agronomic disease models define an FHB-conducive day when three conditions are simultaneously met on the same calendar day:

1. **Temperature (`tas`)**: Daily mean near-surface air temperature is between 15°C and 30°C (inclusive).
2. **Relative humidity (`hurs`)**: Daily mean relative humidity is at least 90%.
3. **Precipitation (`pr`)**: Daily precipitation accumulation strictly exceeds 0 mm.

We will implement this calculation first using the **`xclim` extension framework** wrapped for **earthkit**, and then demonstrate an equivalent implementation using pure **earthkit** and **xarray** operations.

In [1]:
from typing import Any

import earthkit.data as ekd
import numpy as np
import xarray as xr
from xclim.core.indicator import Daily
from xclim.core.units import convert_units_to, declare_units, rate2amount, to_agg_units

from earthkit.climate.utils.decorators import wrap_indicator

### Define the compute function

The compute function defines the core scientific calculation. We use `xclim` utilities for unit-safe conversion, daily precipitation rate-to-amount conversion, and aggregation unit handling. The three threshold masks are combined with a logical AND (`&`), ensuring that all qualifying conditions must be met on the same calendar day.

In [2]:
@declare_units(
    tas="[temperature]",
    hurs="[]",
    pr="[precipitation]",
    tas_lower="[temperature]",
    tas_upper="[temperature]",
    hurs_lower="[]",
    pr_lower="[length]",
)
def fusarium_head_blight_conducive_days_index(
    tas: "xr.DataArray",
    hurs: "xr.DataArray",
    pr: "xr.DataArray",
    tas_lower: "str" = "15 degC",
    tas_upper: "str" = "30 degC",
    hurs_lower: "str" = "90 %",
    pr_lower: "str" = "0 mm",
    freq: "str" = "YS",
) -> "xr.DataArray":
    """Return the number of Fusarium head blight conducive days.

    A day is conducive when mean temperature is within the closed interval
    [tas_lower, tas_upper], relative humidity is at least hurs_lower, and
    daily precipitation is strictly greater than pr_lower.

    Parameters
    ----------
    tas : xarray.DataArray
        Daily mean near-surface air temperature.
    hurs : xarray.DataArray
        Daily mean relative humidity.
    pr : xarray.DataArray
        Daily precipitation flux or rate.
    tas_lower : str
        Inclusive lower temperature threshold.
    tas_upper : str
        Inclusive upper temperature threshold.
    hurs_lower : str
        Inclusive lower relative-humidity threshold.
    pr_lower : str
        Exclusive lower daily precipitation threshold.
    freq : str
        Aggregation frequency; ``YS`` produces one result per calendar year.

    Returns
    -------
    xarray.DataArray
        Number of FHB-conducive days for each period.
    """
    tas_c = convert_units_to(tas, "degC")
    hurs_pct = convert_units_to(hurs, "%")
    pr_amount = rate2amount(pr, out_units="mm")

    lower_t = convert_units_to(tas_lower, tas_c)
    upper_t = convert_units_to(tas_upper, tas_c)
    lower_rh = convert_units_to(hurs_lower, hurs_pct)
    lower_pr = convert_units_to(pr_lower, pr_amount)

    conducive = (tas_c >= lower_t) & (tas_c <= upper_t) & (hurs_pct >= lower_rh) & (pr_amount > lower_pr)

    out = conducive.resample(time=freq).sum()
    out = to_agg_units(out, tas, op="count")
    out.attrs.pop("standard_name", None)
    return out.rename("fhb_conducive_days")

### Create the xclim indicator

The `xclim` layer adds daily-frequency validation, missing-value checks, dataset variable lookup, and CF-compliant output metadata. Here, we instantiate xclim's `Daily` base class.

In [3]:
fhb_conducive_days_xclim = Daily(
    realm="atmos",
    identifier="fhb_conducive_days",
    compute=fusarium_head_blight_conducive_days_index,
    title="Fusarium head blight conducive days",
    abstract="Yearly number of wet, very humid days with temperatures conducive to Fusarium head blight.",
    var_name="fhb_conducive_days",
    long_name="Fusarium head blight conducive days",
    description=(
        "Number of days where {tas_lower} <= T <= {tas_upper}, relative humidity "
        ">= {hurs_lower}, and precipitation > {pr_lower}."
    ),
    units="d",
    cell_methods="time: sum over days",
)

### Add the earthkit indicator wrapper

The `wrap_indicator` decorator allows the indicator to transparently accept multiple data structures, including `xarray.DataArray`, `xarray.Dataset`, and `earthkit-data` objects:

In [4]:
fusarium_head_blight_conducive_days = wrap_indicator(fhb_conducive_days_xclim)

### Load sample data with earthkit-data

We load a synthetic sample dataset registered with the `earthkit-climate-sample` source plugin. The dataset covers two complete years (2020 and 2021) of daily mean temperature (`tas`), relative humidity (`hurs`), and precipitation (`pr`). It contains:

- Qualifying days where all three conditions are met (3 days in 2020, 4 days in 2021).
- Controlled "near misses" where exactly one condition fails (e.g. temperature too high, humidity too low, or no rain), confirming that the logical combiner is strictly verified.

In [5]:
# Load the sample dataset using earthkit-data
ds = ekd.from_source("earthkit-climate-sample", "fusarium-input-timeseries").to_xarray()

# Inspect the qualifying days and near-misses in tabular form
sample_dates = [
    "2020-06-10",
    "2020-06-11",
    "2020-06-20",  # 2020 qualifying days
    "2021-05-15",
    "2021-05-16",
    "2021-05-17",
    "2021-05-30",  # 2021 qualifying days
    "2020-06-25",  # Near miss: T = 31°C (> 30°C)
    "2021-06-01",  # Near miss: RH = 85% (< 90%)
    "2021-06-02",  # Near miss: pr = 0 mm (no rain)
]
ds.sel(time=sample_dates).to_dataframe()

/home/cuadradot/predictia-projects/git/c3s-indices/earthkit-climate/src/earthkit/climate/_sample_source.py:314: UserWarning: earthkit-climate-sample datasets are made available for demonstration purposes only. Files are not guaranteed to be available long-term and may change over time. Please use official channels to obtain the contained datasets reliably for other purposes.
  warnings.warn(


,tas,hurs,pr
time,,,
2020-06-10,20.0,95.0,1.0
2020-06-11,20.0,95.0,1.0
2020-06-20,20.0,95.0,1.0
2021-05-15,20.0,95.0,1.0
2021-05-16,20.0,95.0,1.0
2021-05-17,20.0,95.0,1.0
2021-05-30,20.0,95.0,1.0
2020-06-25,31.0,95.0,1.0
2021-06-01,20.0,85.0,1.0


### Compute and verify yearly FHB-conducive days

We compute the indicator on `ds` and verify that only days satisfying all three conditions are counted (3 days in 2020, 4 days in 2021), while all near misses are excluded:

In [6]:
fhb_days = fusarium_head_blight_conducive_days(ds=ds)

np.testing.assert_array_equal(fhb_days.values, [3, 4])
np.testing.assert_array_equal(fhb_days.time.dt.year.values, [2020, 2021])
assert fhb_days.attrs["units"] == "d"

fhb_days.to_dataframe()

,fhb_conducive_days
time,
2020-01-01,3.0
2021-01-01,4.0


> **Operational note**: This indicator is independent of xclim's pre-existing catalogue. While `xclim` provides the units, validation, missing-value handling, and metadata infrastructure, the scientific logic resides in our compute function. Before operational deployment, specific thresholds and crop growth stages should be calibrated for the target cultivar, pathogen population, and geographic region.

### Implement the indicator without xclim

We can also implement the indicator using only **earthkit** utilities and native `xarray` methods:

- `wrap_indicator`: Translates supported earthkit inputs to `xarray`.
- `earthkit.utils.units.convert_units`: Normalises both input data arrays and unit-bearing thresholds.
- `xarray`: Evaluates the boolean condition masks.
- `xarray.DataArray.resample`: Computes the annual aggregation.

The function signature mirrors the xclim-backed version. Missing values are handled strictly: `skipna=False` ensures that if any input day is missing, the corresponding annual sum is marked as missing (`NaN`).

In [7]:
from earthkit.utils.units import convert_units as ek_convert_units


def _resolve_dataarray(
    value: xr.DataArray | str,
    ds: xr.Dataset | None,
    argument: str,
) -> xr.DataArray:
    """Resolve a DataArray or a variable name from a Dataset."""
    if isinstance(value, xr.DataArray):
        return value
    if ds is None:
        raise ValueError(f"{argument!r} is a variable name, so ds must be supplied")
    try:
        return ds[value]
    except KeyError as exc:
        raise KeyError(f"Dataset has no variable {value!r} for {argument!r}") from exc


def _threshold_magnitude(value: Any, target_units: str) -> float:
    """Convert a numeric or '<magnitude> <units>' threshold."""
    if isinstance(value, str):
        magnitude, source_units = value.strip().split(maxsplit=1)
        return float(
            ek_convert_units(
                float(magnitude),
                target_units=target_units,
                source_units=source_units,
            )
        )
    return float(value)


@wrap_indicator
def fusarium_head_blight_conducive_days_earthkit(
    tas: xr.DataArray | str = "tas",
    hurs: xr.DataArray | str = "hurs",
    pr: xr.DataArray | str = "pr",
    ds: xr.Dataset | Any = None,
    *,
    tas_lower: Any = "15 degC",
    tas_upper: Any = "30 degC",
    hurs_lower: Any = "90 %",
    pr_lower: Any = "0 mm",
    freq: str = "YS",
    **kwargs: Any,
) -> xr.DataArray:
    """Compute yearly FHB-conducive days without xclim."""
    tas = _resolve_dataarray(tas, ds, "tas")
    hurs = _resolve_dataarray(hurs, ds, "hurs")
    pr = _resolve_dataarray(pr, ds, "pr")

    tas_c = ek_convert_units(tas, target_units="degC")
    hurs_pct = ek_convert_units(hurs, target_units="%")
    pr_mm_per_day = ek_convert_units(pr, target_units="mm/day")

    lower_t = _threshold_magnitude(tas_lower, "degC")
    upper_t = _threshold_magnitude(tas_upper, "degC")
    lower_rh = _threshold_magnitude(hurs_lower, "%")
    lower_pr = _threshold_magnitude(pr_lower, "mm")

    conducive = (tas_c >= lower_t) & (tas_c <= upper_t) & (hurs_pct >= lower_rh) & (pr_mm_per_day > lower_pr)
    valid = tas.notnull() & hurs.notnull() & pr.notnull()
    conducive = conducive.where(valid)

    out = conducive.resample(time=freq).sum(skipna=False)
    out = out.rename("fhb_conducive_days")

    source_cell_methods = " ".join(
        f"{argument}: {data.attrs['cell_methods']}"
        for argument, data in (("tas", tas), ("hurs", hurs), ("pr", pr))
        if data.attrs.get("cell_methods")
    )
    description = (
        f"Number of days where {tas_lower} <= T <= {tas_upper}, relative humidity "
        f">= {hurs_lower}, and precipitation > {pr_lower}."
    ).capitalize()
    out.attrs = {
        "cell_methods": f"{source_cell_methods} time: sum over days".strip(),
        "units": "d",
        "long_name": "Fusarium head blight conducive days",
        "description": description,
    }
    return out

### Compare the two approaches

Both implementations produce identical numerical values, coordinates, and dimensions when applied to the same dataset:

In [8]:
fhb_days_earthkit = fusarium_head_blight_conducive_days_earthkit(ds=ds)

np.testing.assert_array_equal(fhb_days_earthkit.values, [3, 4])
assert fhb_days_earthkit.equals(fhb_days)

# equals ignores attrs. identical also checks the name and all remaining attrs.
fhb_days_xclim_stable = fhb_days.copy()
fhb_days_xclim_stable.attrs.pop("history", None)
assert fhb_days_earthkit.identical(fhb_days_xclim_stable)

# With skipna=False passed to the sum, one missing daily input invalidates that year's count.
ds_with_missing = ds.copy(deep=True)
ds_with_missing["tas"].loc[{"time": "2020-06-10"}] = np.nan
fhb_days_with_missing = fusarium_head_blight_conducive_days_earthkit(ds=ds_with_missing)
assert np.isnan(fhb_days_with_missing.sel(time="2020").item())
assert fhb_days_with_missing.sel(time="2021").item() == 4

xr.Dataset({
    "xclim_backed": fhb_days,
    "earthkit_only": fhb_days_earthkit,
}).to_dataframe()

,xclim_backed,earthkit_only
time,,
2020-01-01,3.0,3.0
2021-01-01,4.0,4.0


> **Summary comparison**: 
> - The **standalone earthkit** approach is concise, transparent, and gives complete control over the calculation pipeline.
> - The **`xclim`-backed** approach provides formal indicator objects, standardised parameter introspection, configurable missing-value policies, strict calendar validation, and automated CF metadata.
>
> Both approaches are fully compatible with the **earthkit** ecosystem.